# TP 2 - GRU pour la Prevision de Series Temporelles

**Seance 4 - Architectures Sequentielles Avancees** - Golden Collar

## Objectifs
1. Preparer une serie temporelle (**sliding window**)
2. Entrainer un **GRU** (Keras) et le comparer a un **LSTM** a architecture identique
3. Decouvrir **TimesFM** (foundation model, prevision zero-shot)
4. Comparer les trois **sur la meme tache** (1-pas-en-avant)

## GRU vs LSTM
| Critere | LSTM | GRU |
|---|---|---|
| Portes | 3 (Forget, Input, Output) | 2 (Reset, Update) |
| Cell state separe | Oui | Non |
| Parametres | ~ 4d(d+n+1) | ~ 3d(d+n+1) |
| Vitesse | plus lent | **plus rapide** |

$$h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t \quad \text{(interrupteur continu)}$$

---
## Partie 1 - Configuration

In [ ]:
!pip install -q ucimlrepo

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GRU, LSTM, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from ucimlrepo import fetch_ucirepo

tf.random.set_seed(42)
np.random.seed(42)
print("TensorFlow :", tf.__version__, "| GPU :", tf.config.list_physical_devices('GPU') or "aucun")
# Si vous etes sur Mac (GPU Metal instable), decommentez la ligne suivante pour forcer le CPU :
# tf.config.set_visible_devices([], 'GPU')

---
## Partie 2 - Dataset reel : consommation electrique d'un foyer (UCI)

Mesures **a la minute** (2006-2010), cible `Global_active_power` (puissance active, en **kW**),
**agregee a la journee** (moyenne).

In [ ]:
dataset = fetch_ucirepo(id=235)
raw = dataset.data.features.copy()
raw['datetime'] = pd.to_datetime(raw['Date'].astype(str) + ' ' + raw['Time'].astype(str),
                                 format='%d/%m/%Y %H:%M:%S', errors='coerce')
raw = raw.set_index('datetime')
raw['Global_active_power'] = pd.to_numeric(raw['Global_active_power'], errors='coerce')

df = raw['Global_active_power'].resample('D').mean().dropna().rename('consumption').to_frame()
print(f"{len(df)} jours | {df.index[0].date()} -> {df.index[-1].date()}")
print(df.describe().round(3))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))
axes[0].plot(df.index, df['consumption'], color='steelblue', lw=0.8)
axes[0].set_title('Consommation quotidienne - Global Active Power (kW)')
axes[0].set_ylabel('puissance (kW)'); axes[0].grid(alpha=.3)
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
zoom = df['2009-06':'2009-08']
axes[1].plot(zoom.index, zoom['consumption'], color='#FF5722', lw=1.2, marker='o', ms=2)
axes[1].set_title('Zoom juin-aout 2009 (saisonnalite hebdomadaire)')
axes[1].set_ylabel('puissance (kW)'); axes[1].grid(alpha=.3)
plt.xticks(rotation=45); plt.tight_layout(); plt.show()

---
## Partie 3 - Sliding window

A partir de T observations passees, predire la suivante (1-pas-en-avant).

In [ ]:
scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(df[['consumption']].values)
print(f"normalise : min={data_scaled.min():.3f}, max={data_scaled.max():.3f}")

def create_sequences(data, window, horizon=1):
    X, y = [], []
    for i in range(len(data) - window - horizon + 1):
        X.append(data[i : i+window])
        y.append(data[i+window : i+window+horizon, 0])
    return np.array(X), np.array(y)

WINDOW_SIZE, HORIZON = 30, 1
X, y = create_sequences(data_scaled, WINDOW_SIZE, HORIZON)
print("X :", X.shape, "| y :", y.shape)

In [ ]:
# Split CHRONOLOGIQUE (pas de shuffle pour une serie temporelle)
n = len(X)
i_tr, i_va = int(n*0.70), int(n*0.85)
X_train, y_train = X[:i_tr], y[:i_tr]
X_val,   y_val   = X[i_tr:i_va], y[i_tr:i_va]
X_test,  y_test  = X[i_va:], y[i_va:]
print(f"train {len(X_train)} | val {len(X_val)} | test {len(X_test)}")

---
## Partie 4 - GRU et LSTM (architecture identique)

Un seul `build_rnn(kind=...)` : seule la cellule change -> comparaison equitable.

In [ ]:
def build_rnn(kind='gru', window=WINDOW_SIZE, n_features=1, units=64):
    Cell = GRU if kind == 'gru' else LSTM
    inputs  = Input(shape=(window, n_features))
    x = Cell(units, dropout=0.2)(inputs)     # recurrent_dropout evite (desactive cuDNN)
    x = Dropout(0.2)(x)
    x = Dense(32, activation='relu')(x)
    out = Dense(1)(x)
    return Model(inputs, out, name=f'{kind}_forecaster')

def train(model):
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='mse', metrics=['mae'])
    cbs = [EarlyStopping('val_loss', patience=10, restore_best_weights=True, verbose=1),
           ReduceLROnPlateau('val_loss', factor=0.5, patience=5, verbose=0)]
    return model.fit(X_train, y_train, validation_data=(X_val, y_val),
                     epochs=100, batch_size=32, callbacks=cbs, verbose=1)

In [ ]:
model_gru = build_rnn('gru'); model_gru.summary()

# Parametres GRU : formule theorique 3*d*(d+n+1).
# Keras utilise reset_after=True (biais separes entree/recurrence) -> 3*(d*(d+n) + 2d), un peu plus.
d, nfeat = 64, 1
theo  = 3 * d * (d + nfeat + 1)
keras = 3 * (d * (d + nfeat) + 2 * d)
real  = next(l for l in model_gru.layers if l.__class__.__name__ == 'GRU').count_params()
print(f"\nGRU - theorique {theo:,} | Keras reset_after {keras:,} | reel {real:,}")

In [ ]:
history_gru = train(model_gru)

---
## Partie 5 - LSTM (comparaison)

In [ ]:
model_lstm = build_rnn('lstm'); model_lstm.summary()
print(f"\nGRU  : {model_gru.count_params():,} params")
print(f"LSTM : {model_lstm.count_params():,} params (ratio {model_lstm.count_params()/model_gru.count_params():.2f}x)")

In [ ]:
history_lstm = train(model_lstm)

---
## Partie 6 - Comparaison GRU vs LSTM

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5))
for h, name, c in [(history_gru, 'GRU', '#4CAF50'), (history_lstm, 'LSTM', '#2196F3')]:
    a1.plot(h.history['loss'], c=c, label=f'{name} train'); a1.plot(h.history['val_loss'], c=c, ls='--', label=f'{name} val')
    a2.plot(h.history['mae'], c=c, label=f'{name} train'); a2.plot(h.history['val_mae'], c=c, ls='--', label=f'{name} val')
a1.set_title('Loss (MSE)'); a1.set_xlabel('epoch'); a1.legend(); a1.grid(alpha=.3)
a2.set_title('MAE'); a2.set_xlabel('epoch'); a2.legend(); a2.grid(alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
def evaluate(model, name):
    y_pred = scaler.inverse_transform(model.predict(X_test, verbose=0)).flatten()
    y_true = scaler.inverse_transform(y_test).flatten()
    m = {'MAE':  mean_absolute_error(y_true, y_pred),
         'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
         'MAPE': np.mean(np.abs((y_true - y_pred) / y_true)) * 100}
    print(f"{name:6s} MAE={m['MAE']:.3f} kW  RMSE={m['RMSE']:.3f} kW  MAPE={m['MAPE']:.2f}%")
    return y_pred, y_true, m

y_pred_gru,  y_true, metrics_gru  = evaluate(model_gru, 'GRU')
y_pred_lstm, _,      metrics_lstm = evaluate(model_lstm, 'LSTM')

In [ ]:
test_dates = df.index[WINDOW_SIZE + i_va : WINDOW_SIZE + i_va + len(y_true)]
z = 60
plt.figure(figsize=(14, 5))
plt.plot(test_dates[:z], y_true[:z], 'k-o', ms=3, label='reel')
plt.plot(test_dates[:z], y_pred_gru[:z], '-s', ms=3, c='#4CAF50', alpha=.8, label='GRU')
plt.plot(test_dates[:z], y_pred_lstm[:z], '-^', ms=3, c='#2196F3', alpha=.8, label='LSTM')
plt.title(f'{z} premiers jours de test (1-pas)'); plt.ylabel('puissance (kW)')
plt.legend(); plt.grid(alpha=.3); plt.xticks(rotation=45); plt.tight_layout(); plt.show()

---
## Partie 7 - TimesFM (foundation model, zero-shot)

**Comparaison equitable :** on evalue TimesFM en **rolling 1-pas** (comme GRU/LSTM) : pour chaque
jour cible, on fournit le contexte reel precedent et on prevoit 1 pas. Pas de normalisation ni
d'entrainement cote TimesFM (zero-shot).

In [ ]:
# Installation (a executer une fois) :
# !pip install "git+https://github.com/google-research/timesfm.git#egg=timesfm[torch]"

In [ ]:
USE_TIMESFM = False
try:
    import torch, timesfm
    torch.set_float32_matmul_precision("high")
    model_tfm = timesfm.TimesFM_2p5_200M_torch.from_pretrained("google/timesfm-2.5-200m-pytorch")
    model_tfm.compile(timesfm.ForecastConfig(
        max_context=512, max_horizon=64, normalize_inputs=True,
        use_continuous_quantile_head=True, force_flip_invariance=True,
        infer_is_positive=True, fix_quantile_crossing=True))
    USE_TIMESFM = True
    print("TimesFM 2.5 charge.")
except Exception as e:
    print("TimesFM indisponible -> section ignoree (pas de resultats simules).", type(e).__name__)

In [ ]:
metrics_tfm, y_pred_tfm = None, None
if USE_TIMESFM:
    series = df['consumption'].values.astype(np.float32)   # serie brute (kW)
    start = WINDOW_SIZE + i_va                              # 1er indice de test
    CTX = 256

    # Rolling 1-pas : contexte reel se terminant juste avant chaque cible
    contexts = [series[max(0, start+k-CTX) : start+k].tolist() for k in range(len(y_true))]
    preds = []
    for i in range(0, len(contexts), 64):
        pf, _ = model_tfm.forecast(horizon=1, inputs=contexts[i:i+64])
        preds.extend(np.asarray(pf, dtype=np.float32)[:, 0])
    y_pred_tfm = np.array(preds)

    metrics_tfm = {'MAE':  mean_absolute_error(y_true, y_pred_tfm),
                   'RMSE': np.sqrt(mean_squared_error(y_true, y_pred_tfm)),
                   'MAPE': np.mean(np.abs((y_true - y_pred_tfm) / y_true)) * 100}
    print(f"TimesFM (zero-shot, rolling 1-pas) MAE={metrics_tfm['MAE']:.3f} kW "
          f"RMSE={metrics_tfm['RMSE']:.3f} kW MAPE={metrics_tfm['MAPE']:.2f}%")
else:
    print("TimesFM non evalue.")

In [ ]:
if USE_TIMESFM:
    z = 100
    plt.figure(figsize=(14, 6))
    plt.plot(test_dates[:z], y_true[:z], 'k-', lw=1.5, label='reel')
    plt.plot(test_dates[:z], y_pred_gru[:z],  c='#4CAF50', alpha=.7, label='GRU')
    plt.plot(test_dates[:z], y_pred_lstm[:z], c='#2196F3', alpha=.7, label='LSTM')
    plt.plot(test_dates[:z], y_pred_tfm[:z],  c='#FF5722', alpha=.7, ls='--', label='TimesFM (zero-shot)')
    plt.title('GRU vs LSTM vs TimesFM - 100 jours de test (1-pas)'); plt.ylabel('puissance (kW)')
    plt.legend(); plt.grid(alpha=.3); plt.xticks(rotation=45); plt.tight_layout(); plt.show()

---
## Partie 8 - Recapitulatif

In [ ]:
cols = [('GRU', metrics_gru), ('LSTM', metrics_lstm)]
if USE_TIMESFM:
    cols.append(('TimesFM', metrics_tfm))

print(f"{'Metrique':<12}" + ''.join(f"{n:>14}" for n, _ in cols))
print('-' * (12 + 14*len(cols)))
for k in ['MAE', 'RMSE', 'MAPE']:
    unit = '%' if k == 'MAPE' else 'kW'
    print(f"{k+' ('+unit+')':<12}" + ''.join(f"{m[k]:>14.3f}" for _, m in cols))
print(f"{'Entrainement':<12}" + f"{'oui':>14}{'oui':>14}" + ("{:>14}".format('non (zero-shot)') if USE_TIMESFM else ""))
print(f"{'Parametres':<12}" + f"{model_gru.count_params():>14,}{model_lstm.count_params():>14,}" + ("{:>14}".format('200M') if USE_TIMESFM else ""))

print("\nNote : les trois sont evalues a horizon 1 (rolling), donc comparables.")
print("GRU ~ LSTM ici ; GRU plus rapide (moins de params). TimesFM = zero-shot, sans preparation.")

---
## Partie 9 - Analyse des erreurs

In [ ]:
series_err = [('GRU', y_true - y_pred_gru, '#4CAF50'), ('LSTM', y_true - y_pred_lstm, '#2196F3')]
if USE_TIMESFM:
    series_err.append(('TimesFM', y_true - y_pred_tfm, '#FF5722'))

fig, axes = plt.subplots(1, len(series_err), figsize=(5*len(series_err), 4))
if len(series_err) == 1: axes = [axes]
for ax, (name, err, c) in zip(axes, series_err):
    ax.hist(err, bins=40, color=c, edgecolor='white', alpha=.8)
    ax.axvline(0, color='black', ls='--', lw=1)
    ax.set_title(f'Erreurs - {name}'); ax.set_xlabel('erreur (kW)')
    ax.text(0.02, 0.95, f"mu={np.mean(err):.3f}\nsigma={np.std(err):.3f}",
            transform=ax.transAxes, va='top', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='white', alpha=.8))
plt.tight_layout(); plt.show()

## Exercices
1. **Stacked GRU** (2 couches, `return_sequences=True` sur la 1re).
2. **Multi-step** : `HORIZON=7` et adapter la couche de sortie.
3. **Multi-varie** : ajouter jour de semaine / mois / week-end (`n_features`).
4. **BiGRU** pour le forecasting : pertinent ? (rappel : le bidirectionnel exige la sequence
   complete -> pas de prediction autoregressive).

## References
- Cho et al. (2014, GRU) ; Hochreiter & Schmidhuber (1997, LSTM) ;
  Das et al. (2024, TimesFM, ICML).